# 01. 문서 요약 Tool — PDF 전처리 + `summary.py`
**Day 4**

> 수업용 문서: **`samples/Language_Models.pdf`**
> (언어 모델 개요 자료 — 긴 PDF 요약 실습에 적합)

**파이프라인:** `PDF → PyMuPDF 추출 → 전처리 → OpenAI 요약 → 저장`

---
## 0. 설치

In [ ]:
#!pip install pymupdf openai python-dotenv

In [ ]:
#!pip install pymupdf

In [12]:
import os
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

True

In [13]:
pdf_path = os.path.join(os.getcwd(),"samples/Language_models.pdf")

---
## 1. PyMuPDF로 `Language_Models.pdf` 텍스트 추출

In [14]:
import pymupdf

doc = pymupdf.open(pdf_path)

In [15]:
doc

Document('d:\autornd\SK Autonomous R&D\실습\4일차\samples/Language_models.pdf')

In [16]:
i=0
for page in doc:
    text = page.get_text()
    print(text)
    i+=1
    if i==5:
        break

Language Models are Few-Shot Learners
Tom B. Brown∗
Benjamin Mann∗
Nick Ryder∗
Melanie Subbiah∗
Jared Kaplan†
Prafulla Dhariwal
Arvind Neelakantan
Pranav Shyam
Girish Sastry
Amanda Askell
Sandhini Agarwal
Ariel Herbert-Voss
Gretchen Krueger
Tom Henighan
Rewon Child
Aditya Ramesh
Daniel M. Ziegler
Jeffrey Wu
Clemens Winter
Christopher Hesse
Mark Chen
Eric Sigler
Mateusz Litwin
Scott Gray
Benjamin Chess
Jack Clark
Christopher Berner
Sam McCandlish
Alec Radford
Ilya Sutskever
Dario Amodei
OpenAI
Abstract
Recent work has demonstrated substantial gains on many NLP tasks and benchmarks by pre-training
on a large corpus of text followed by ﬁne-tuning on a speciﬁc task. While typically task-agnostic
in architecture, this method still requires task-speciﬁc ﬁne-tuning datasets of thousands or tens of
thousands of examples. By contrast, humans can generally perform a new language task from only
a few examples or from simple instructions – something which current NLP systems still largely
struggle

In [17]:
full_text=''
for page in doc:
    text = page.get_text()
    full_text+=text + '\n------------------------\n'


In [18]:
pdf_file_name = os.path.basename(pdf_path)

In [19]:
pdf_file_name = os.path.splitext(pdf_file_name)[0]
txt_file_path = os.path.join(os.getcwd(),'samples',f"{pdf_file_name}.txt")

In [20]:
with open(txt_file_path, 'w', encoding='utf-8') as f:
    f.write(full_text)

---
## 2. 문서 요약

In [21]:
api_key = os.getenv('OPENAI_API_KEY')

def summarize_txt(file_path):
    client = OpenAI(api_key=api_key)
    with open(file_path, 'r', encoding = 'utf-8') as f:
        txt = f.read()

    system_prompt = f'''
    너는 다음 글을 요약하는 봇이다. 아래 글을 읽고, 

    작성해야 하는 포맷은 다음과 같음
    # 제목

    ## 저자의 문제 인식 및 주장 (15문장 이내)

    ## 저자 소개


    ============= 이하 텍스트 ================
    {txt[:10000]}

    '''

    response = client.chat.completions.create(
        model = 'gpt-5.4-mini',
        temperature = 0.1,
        messages=[
            {"role":"system","content":system_prompt},
        ]
    )

    return response.choices[0].message.content


In [22]:
summary = summarize_txt(txt_file_path)

In [19]:
print(summary)

# Language Models are Few-Shot Learners

## 저자의 문제 인식 및 주장 (15문장 이내)

기존 NLP는 사전학습 후 파인튜닝으로 큰 성과를 냈지만, 여전히 각 과제마다 수천~수만 개의 라벨 데이터와 별도 파인튜닝이 필요하다는 한계가 있다.  
저자들은 인간처럼 적은 예시나 간단한 지시만으로 새 언어 과제를 수행하는 능력이 현재 NLP 시스템에는 부족하다고 본다.  
이 문제를 해결하는 한 가지 방법으로, 대규모 언어모델을 더 크게 학습시키면 과제별 추가 학습 없이도 few-shot 성능이 크게 향상될 수 있다고 주장한다.  
이를 검증하기 위해 1750억 개 파라미터의 GPT-3를 학습하고, gradient update나 fine-tuning 없이 텍스트 입력만으로 다양한 과제를 수행하게 했다.  
그 결과 번역, 질의응답, 빈칸 채우기, 상식 추론, 간단한 산술 같은 여러 과제에서 강한 성능을 보였다.  
특히 일부 과제에서는 기존 최고 수준의 파인튜닝 방식과 경쟁할 만한 성능에 도달했다.  
또한 모델 규모가 커질수록 문맥 속 예시를 더 효율적으로 활용하는 in-context learning 능력이 좋아진다는 점을 보였다.  
다만 모든 과제에서 성공한 것은 아니며, 일부 데이터셋에서는 여전히 성능이 부족했다.  
또한 대규모 웹 코퍼스로 학습한 탓에 데이터 오염이나 방법론적 문제도 존재한다고 지적했다.  
더 나아가 GPT-3가 사람이 구분하기 어려운 수준의 뉴스 기사도 생성할 수 있어, 오용 가능성과 사회적 영향도 함께 논의했다.  

## 저자 소개

이 논문의 저자는 OpenAI 연구진을 중심으로 한 다수의 연구자들이다.  
대표 저자는 Tom B. Brown, Benjamin Mann, Nick Ryder, Melanie Subbiah 등이며,  
Jared Kaplan, Prafulla Dhariwal, Arvind Neelakantan, Pranav Shyam, Ilya Sutskever, Dario Amod

## 실습 : 함수 생성

- input : pdf 파일 경로 
- output : pdf 내용을 요약한 summary.txt 파일 생성 및 summary return

- 파일명: pdf_summary.py

In [25]:
from pdf_summary import summarize_pdf

summary_result = summarize_pdf(pdf_path, save=True)

In [26]:
print(summary_result)

# Language Models are Few-Shot Learners

## 저자의 문제 인식 및 주장 (15문장 이내)

저자들은 기존 NLP 방식이 대규모 사전학습 뒤에도 특정 과제별 미세조정 데이터가 많이 필요하다는 점을 문제로 본다.  
이 방식은 과제마다 수천~수십만 개의 라벨 데이터가 필요해, 새로운 언어 과제에 빠르게 적용하기 어렵다.  
또한 좁은 데이터 분포에 맞춘 미세조정은 모델이 편향된 상관관계에 과적합할 위험이 있다.  
반면 인간은 적은 예시나 간단한 지시만으로도 새로운 언어 과제를 수행할 수 있다.  
저자들은 이런 인간의 적응 능력을 NLP 시스템도 가져야 한다고 주장한다.  
이를 위해 대규모 언어모델이 훈련 중 다양한 패턴과 능력을 학습하고, 추론 시 문맥만으로 과제를 파악하는 “in-context learning”이 핵심이라고 본다.  
이 논문은 GPT-3라는 1750억 개 파라미터의 초대형 언어모델을 사용해 이 가능성을 검증한다.  
GPT-3는 그래디언트 업데이트나 미세조정 없이, 텍스트로만 과제 설명과 몇 개의 예시를 받아 수행한다.  
실험 결과 GPT-3는 번역, 질문응답, 빈칸 채우기, 상식 추론, 간단한 산술 등 여러 과제에서 강한 성능을 보였다.  
특히 일부 과제에서는 기존의 미세조정 기반 최고 성능과 비슷한 수준까지 도달했다.  
동시에 몇몇 데이터셋에서는 여전히 성능이 부족했고, 대규모 웹 코퍼스 학습으로 인한 방법론적 문제도 확인했다.  
또한 GPT-3가 사람이 작성한 뉴스 기사와 구분하기 어려운 수준의 문장을 생성할 수 있음을 보였다.  
저자들은 이 결과가 언어모델의 유용성을 크게 확장하지만, 오용·편향·에너지 사용 같은 사회적 영향도 함께 고려해야 한다고 강조한다.  

## 저자 소개

이 논문의 저자들은 OpenAI 연구진을 중심으로 구성되어 있으며, 대표 저자로 Tom B. Brown, Benjamin Mann, Nick Ryder, Melanie Subbiah 등이 있다.  
공동 저자에는